In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

In [4]:
file_path = 'Data/dane.csv'

heart_test = pd.read_csv('Data/heart_test.csv')
heart_train = pd.read_csv('Data/heart_train.csv')
mushrooms_test = pd.read_csv('Data/mushrooms_test.csv')
mushrooms_train = pd.read_csv('Data/mushrooms_train.csv')
rice_test = pd.read_csv('Data/rice_test.csv')
rice_train = pd.read_csv('Data/rice_train.csv')
wine_test = pd.read_csv('Data/wine_test.csv')
wine_train = pd.read_csv('Data/wine_train.csv')

datasets = {
    "heart": (heart_train, heart_test),
    "mushrooms": (mushrooms_train, mushrooms_test),
    "rice": (rice_train, rice_test),
    "wine": (wine_train, wine_test)
}

In [5]:
n_random = 100
np.random.seed(42)

# Funkcja 2^x, x ∈ [-10, 10]
def log_uniform(base=2, low=-10, high=10, size=100):
    return base ** np.random.uniform(low, high, size)

from scipy.stats import reciprocal

param_dist = {
    'C': reciprocal(1e-3, 1e3),               # ~ 2^[-10, 10]
    'gamma': reciprocal(1e-3, 1e3),           # ~ 2^[-10, 10]
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [2, 3, 4, 5]      # tylko dla 'poly' i 'sigmoid'
}



In [6]:
# all_results = []

# for name, (train, test) in datasets.items():
#     print(f"Trenuję model dla: {name}")

#     X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
#     X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

#     model = SVC(probability=True)

#     search = RandomizedSearchCV(
#         estimator=model,
#         param_distributions=param_dist,
#         n_iter=10,
#         scoring='roc_auc',
#         cv=5,
#         verbose=1,
#         random_state=42,
#         n_jobs=-1
#     )

#     search.fit(X_train, y_train)

#     cv_results = pd.DataFrame(search.cv_results_)

#     for i, params in enumerate(search.cv_results_['params']):
#         model = SVC(probability=True, random_state=42, **params)
#         model.fit(X_train, y_train)
#         y_proba = model.predict_proba(X_test)[:, 1]
#         test_auc = roc_auc_score(y_test, y_proba)

#         all_results.append({
#             "dataset": name,
#             "params": params,
#             "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
#             "test_roc_auc": test_auc
#         })

# results_df = pd.DataFrame(all_results)

# results_df.to_csv("wyniki_svm.csv", index=False)


In [7]:
wyniki_svm = pd.read_csv('wyniki_svm.csv')

In [10]:
best_per_dataset = (
    wyniki_svm.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

aggregated_params = {}

for col in params_df.columns:
    if pd.api.types.is_numeric_dtype(params_df[col]):
        aggregated_params[col] = params_df[col].mean()
    else:
        aggregated_params[col] = params_df[col].mode().iloc[0]  # najczęstsza wartość

mean_params = pd.Series(aggregated_params)

In [12]:
# mean_params_dict = mean_params.to_dict()
# for param in ["max_depth", "min_samples_leaf", "min_samples_split"]:
#     mean_params_dict[param] = int(round(mean_params_dict[param]))
# mean_results = []

# for name, (train_path, test_path) in datasets.items():
#     train = pd.read_csv(train_path)
#     test = pd.read_csv(test_path)

#     X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
#     X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

#     model = DecisionTreeClassifier(random_state=42, **mean_params_dict)
#     model.fit(X_train, y_train)
#     y_proba = model.predict_proba(X_test)[:, 1]
#     mean_auc = roc_auc_score(y_test, y_proba)

#     mean_results.append({
#         "dataset": name,
#         "mean_test_roc_auc": mean_auc
#     })

# mean_df = pd.DataFrame(mean_results)


,0
0,"{'C': 12.073834860996039, 'degree': 2, 'gamma'..."
1,"{'C': 0.1767016940294795, 'degree': 2, 'gamma'..."
2,"{'C': 12.073834860996039, 'degree': 2, 'gamma'..."
3,"{'C': 12.073834860996039, 'degree': 2, 'gamma'..."


In [ ]:
# results_df = results_df.merge(mean_df, on="dataset")
# results_df["diff_from_mean"] = results_df["mean_test_roc_auc"] - results_df["test_roc_auc"]
# results_df

# results_df.sort_values(by="diff_from_mean",ascending=True).head(20)

# results_df.to_csv("results.csv", index=False)